In [103]:
from spin_lattices import TriangleLattice, SquareLattice, KagomeLattice,  SpinLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
from fast_boolean_analysis import FourierSeries, fourier_expand, keep_largest_n
from lattice_boolean_analysis import LBFFromSpinSystem
from pathlib import Path
import numpy as np
from nn_xors_2023_07_18 import make_dataset, train, MLPBinaryClassifier
from loguru import logger
import torch
from torch.utils.data import random_split, DataLoader
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torch import nn
from misc_utils import make_unpacked_configurations, make_packed_configurations
from parity import popcount
import matplotlib.pyplot as plt
from itertools import product
import pandas as pd

In [71]:
np.unique(np.array([1, 2, 3, 2, 3, 1]), return_inverse=True)

(array([1, 2, 3]), array([0, 1, 2, 1, 2, 0]))

In [108]:
lattice = KagomeLattice(2, 4)
system = HeisenbergJ1J2(lattice, J1=1, J2=0.5, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)
signal = LBFFromSpinSystem(system)
series = fourier_expand(signal)
truncated_series = series.truncate_orbitwise(keep_largest_n(2))

print(np.where(truncated_series.coeffs != 0))
print(popcount(np.where(truncated_series.coeffs != 0)[0].astype('uint64')))
print(truncated_series.coeffs[truncated_series.coeffs != 0])

truncated_series = shuffle_xors(truncated_series)
print(np.where(truncated_series.coeffs != 0))
print(popcount(np.where(truncated_series.coeffs != 0)[0].astype('uint64')))
print(truncated_series.coeffs[truncated_series.coeffs != 0])

2023-07-24 18:01:13.535 | DEBUG    | heisenberg_hamiltonians:__init__:435 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-07-24 18:01:13.537 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=24
2023-07-24 18:01:13.542 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-24 18:01:13.607 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 85662
2023-07-24 18:01:13.616 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:64 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.5-True-1-1.pickle
2023-07-24 18:01:13.618 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:111 - Ground state energy is -39.4742602743
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-24 18:01:13.704 | DEBUG    | fast_boole

(array([  169125,  1217703,  2266341,  4365477,  8153946,  8623269,
       12411738, 14510874, 15559512, 16608090]),)
[ 8 10 10 10 14 10 14 14 14 16]
[-589.21386719   52.30957031   52.30957031   52.30957031   52.30957031
   52.30957031   52.30957031   52.30957031   52.30957031 -589.21386719]
(array([  169125,  1217703,  2266341,  4365477,  8153946,  8623269,
       12411738, 14510874, 15559512, 16608090]),)
[ 8 10 10 10 14 10 14 14 14 16]
[  52.30957031 -589.21386719   52.30957031   52.30957031   52.30957031
   52.30957031   52.30957031 -589.21386719   52.30957031   52.30957031]


In [98]:
lattice = SquareLattice(2 * 3, 4)

In [101]:
eps_train = 0.01
eps_test = 0.1
batch_size = 64
n_hidden = 512
epochs = 100

runs = 20

system = HeisenbergJ1J2(lattice, J1=1, J2=0.5, ground_state_cache_dir=Path("groundstates"))
system.get_eigenstates(1)
signal = LBFFromSpinSystem(system)
series = fourier_expand(signal)

n_spins = system.number_spins
results = []
for keep, transform, run in product(
    [2], [identity, replace_xors_with_random, shuffle_xors], range(runs)
):
    writer = SummaryWriter(
        log_dir=(
            f"experiments/2023_07_20/{datetime.now().strftime('%H_%M_%S')}"
            #            f"_{n_xors=}_{xor_hamming_weight=}_{distance=}_{run=}"
        )
    )
    all_states = system.canonical_basis.states
    sample_states = np.random.choice(
        all_states,
        size=int(len(all_states) * (eps_train + eps_test)),
        replace=False,
    )

    if keep is not None:
        truncated_series = series.truncate_orbitwise(keep_largest_n(keep))
        # non_zero_idxs = np.where(truncated_series.coeffs != 0)[0]
        # non_zero_idxs_shuffled = non_zero_idxs
        # np.random.shuffle(non_zero_idxs_shuffled)
        # truncated_series.coeffs[non_zero_idxs] = truncated_series.coeffs[non_zero_idxs_shuffled]
    else:
        truncated_series = series

    if transform is not None:
        truncated_series = transform(truncated_series)

    dataset = make_dataset(truncated_series, sample_states, n_spins)
    train_dataset, test_dataset = random_split(
        dataset, [eps_train / (eps_train + eps_test), eps_test / (eps_train + eps_test)]
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    net = MLPBinaryClassifier(n_spins, n_hidden)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

    output = train(
        net=net,
        criterion=criterion,
        optimizer=optimizer,
        train_loader=train_loader,
        test_dataset=test_dataset,
        n_epochs=epochs,
        writer=writer,
        break_on_loss=1e-03,
    )
    results.append(
        output
        | {
            "keep": keep,
            "transform": transform.__name__,
            "run": run,
            "non_zero_coeffs": (truncated_series.coeffs != 0).sum(),
            "system": system.get_cache_id(),
        }
    )
    logger.debug(f"Finished training, {output=}")

2023-07-24 17:45:24.883 | DEBUG    | heisenberg_hamiltonians:__init__:435 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-07-24 17:45:24.885 | DEBUG    | heisenberg_hamiltonians:__init__:456 - number_spins=24
2023-07-24 17:45:24.925 | DEBUG    | heisenberg_hamiltonians:__init__:466 - Symmetry group contains 96 elements
2023-07-24 17:45:24.926 | DEBUG    | heisenberg_hamiltonians:__init__:475 - Hilbert space dimension is 15578
2023-07-24 17:45:24.949 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:64 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-SquareLattice6x4-1.0-0.5-True-1-1.pickle
2023-07-24 17:45:24.951 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:111 - Ground state energy is -50.1623937953
2023-07-24 17:45:24.952 | DEBUG    | fast_boolean_analysis:fourier_expand:295 - Finding signal
2023-07-24 17:45:24.954 | DEBUG    | heisenberg_hamiltonians:get_ground_

In [104]:
pd.DataFrame(results)

,loss,train_accuracy,test_accuracy,epoch,keep,transform,run,non_zero_coeffs,system
0,0.071088,0.987205,0.820354,99,2,identity,0,6,HeisenbergJ1J2-SquareLattice6x4-1.0-0.5-True-1
1,0.280877,0.885400,0.608184,99,2,replace_xors_with_random,0,6,HeisenbergJ1J2-SquareLattice6x4-1.0-0.5-True-1
2,0.001967,1.000000,0.983588,99,2,shuffle_xors,0,6,HeisenbergJ1J2-SquareLattice6x4-1.0-0.5-True-1
